In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import plotly.graph_objects as go

from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# data loading
import sys
sys.path.insert(0, '/Users/zzhu/JupyterLab/MarketMaking/_packages')
from tardis_loader import TardisLoader

import nest_asyncio
nest_asyncio.apply()

# plot
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# exchange data info
# https://api.tardis.dev/v1/exchanges/binance

# For generate_combination function
import json
from itertools import product
from typing import List, Dict, Tuple

# For Tardis download error
from urllib.error import HTTPError

# For pandas print setting
pd.set_option('display.width', None)  # Automatically adjust the width of the display
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load the format config
with open("./config/format_cfg.json", "r") as f:
    format_cfg = json.load(f)

# Load Tardis credential
root_dir = './data'
api_key = 'TD.sSGNjEpohBl9i0FV.zz0-s-b6DZ9c2lt.uoNpgBPGaTITrA9.YlW26JKS0PSfoOG.2pEnnkHAnTpOloD.3wMo'

Tardis_data = TardisLoader(root_dir, api_key)

In [ ]:
root_dir = './data'
api_key = 'TD.sSGNjEpohBl9i0FV.zz0-s-b6DZ9c2lt.uoNpgBPGaTITrA9.YlW26JKS0PSfoOG.2pEnnkHAnTpOloD.3wMo'

Tardis_data = TardisLoader(root_dir, api_key)

# Set time and exchanges
start_time = datetime.strptime('2025-05-14', '%Y-%m-%d')
end_time = datetime.strptime('2025-05-14', '%Y-%m-%d')
exchange = 'binance-futures' #'bybit-spot'， ‘binance-futures’
symbol = 'penguusdt' # moodengusdt, komausdt, neirousdt
                    
# Load the data
order_book_25 = Tardis_data.read(start_time, end_time, data_type='book_snapshot_25', exchange=exchange, symbol=symbol)

In [ ]:
# Downsample the original dataset
order_book_25_resample = order_book_25.group_by_dynamic("timestamp", every=f"{1_000_000}i").agg(
    *[pl.first(col) for col in order_book_25.columns if col != "timestamp"]
)

# Span the price and amount col for next filter step
price_cols = [col for col in order_book_25_resample.columns if col.endswith('.price')]
amount_cols = [col for col in order_book_25_resample.columns if col.endswith('.amount')]

ob_price_long = order_book_25_resample.unpivot(on=price_cols, index=['timestamp'], variable_name='price_level', value_name='price')
ob_amount_long = order_book_25_resample.unpivot(on=amount_cols, index=['timestamp'], variable_name='amount_level', value_name='amount')

ob_price_long = ob_price_long.with_columns([
    pl.col("price_level").str.replace("price", "amount").alias("amount_level")
])

orderbooks_long = ob_price_long.join(ob_amount_long, on=['timestamp', 'amount_level'], how = 'left')

orderbooks_liquidity = orderbooks_long.group_by('timestamp').agg(
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9995) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_5bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0005) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_5bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.999) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_10bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.001) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_10bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9985) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_15bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0015) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_15bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.998) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_20bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.002) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_20bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9975) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_25bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0025) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_25bps'),
        ).sort('timestamp')
        
orderbooks_liquidity = orderbooks_liquidity.with_columns(
    *[pl.col(col).ewm_mean(alpha=0.01, adjust=False).alias(col) for col in orderbooks_liquidity.columns if 'liquidity' in col]
).with_columns(
    *[pl.col(col) * -1 for col in orderbooks_liquidity.columns if 'bid' in col]
)


In [ ]:
# Convert to Pandas for Plotly compatibility
df_pandas = orderbooks_liquidity.to_pandas()
df_pandas['timestamp_utc'] = pd.to_datetime(df_pandas['timestamp'], unit='us')

plot_columns = [col for col in df_pandas.columns if 'bps' in col]

# Create an interactive plot
fig = px.line(df_pandas, x="timestamp_utc", y=plot_columns, title="Interactive Price Plot")

# Show the plot
fig.show()

In [ ]:
def generate_combinations(
    data_type: List[str],
    exchange: List[str],
    symbol: List[str],
    format_config: Dict
) -> List[Tuple[str, str, str]]:
    """
    Generate all possible combinations of data_type, exchange, and symbol,
    while formatting exchange and symbol based on the provided format_config.
    """
    combinations = []

    for dt, ex, sym in product(data_type, exchange, symbol):
        # Extract the base exchange name (e.g., 'bybit' from 'bybit-spot')
        base_exchange, market_type = ex.split('-')
        
        # Ensure the base exchange and market type exist in the config
        if base_exchange in format_config and market_type in format_config[base_exchange]:
            # Format the exchange
            formatted_exchange = format_config[base_exchange][market_type]["exchange"]
            
            # Format the symbol
            symbol_format = format_config[base_exchange][market_type]["symbol"]
            formatted_symbol = sym.lower() + symbol_format
            
            # Append the tuple
            combinations.append((dt, formatted_exchange, formatted_symbol))

    return combinations

In [ ]:
def liquidity_examine(start_time_str: str, end_time_str: str, exchanges: list[str], symbols: list[str], price_range: list[float]) -> pl.DataFrame:
    '''
    granularity='50ms'
    '''
    # Download the dataset - Nov 7 vs Today
    start_time = datetime.strptime(start_time_str, "%Y-%m-%d")
    end_time = datetime.strptime(end_time_str, "%Y-%m-%d")

    # Convert the exchanges' and symbols' format according to format_cfg
    combinations = generate_combinations(data_types, exchanges, symbols, format_cfg)

    venues = []
    
    for comb in combinations:
        try:
            # Tardis_data.download(start_time, end_time, data_type=comb[0], exchange=comb[1], symbol=comb[2])
            data = Tardis_data.read(start_time, end_time, data_type=comb[0], exchange=comb[1], symbol=comb[2])            
            data = data.group_by_dynamic("timestamp", every=f"{1_000_000}i").agg(
                *[pl.first(col) for col in data.columns if col != "timestamp"]
            )
            venues.append(data)

        except HTTPError:
            print(f'{comb[2]} {comb[0]} not avaliable on {comb[1]}.')
        except Exception as e:
            print(e)

    # Join the bbo data from 3 exchanges to initial timestamp
    venues_liquidity = pl.DataFrame()
    
    for venue in venues:
        venue_name = venue[0]['exchange'][0]
        # Span the price and amount col for next filter step
        order_book_25_resample = venue.filter((pl.col("asks[0].price") >= price_range[0]) & (pl.col("asks[0].price") <= price_range[1]))
        
        price_cols = [col for col in order_book_25_resample.columns if col.endswith('.price')]
        amount_cols = [col for col in order_book_25_resample.columns if col.endswith('.amount')]

        ob_price_long = order_book_25_resample.unpivot(on=price_cols, index=['timestamp'], variable_name='price_level', value_name='price')
        ob_amount_long = order_book_25_resample.unpivot(on=amount_cols, index=['timestamp'], variable_name='amount_level', value_name='amount')

        ob_price_long = ob_price_long.with_columns([
            pl.col("price_level").str.replace("price", "amount").alias("amount_level")
        ])

        orderbooks_long = ob_price_long.join(ob_amount_long, on=['timestamp', 'amount_level'], how = 'left')
        
        orderbooks_liquidity = orderbooks_long.group_by('timestamp').agg(
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9995) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_5bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0005) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_5bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.999) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_10bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.001) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_10bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9985) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_15bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0015) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_15bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.998) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_20bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.002) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_20bps'),
            (pl.col('amount').filter((pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 0.9975) & 
                                    (pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('bid_liquidity_in_25bps'),
            (pl.col('amount').filter((pl.col('price') <= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2) * 1.0025) & 
                                    (pl.col('price') >= ((pl.col('price').filter(pl.col('price_level') == "asks[0].price").first() + pl.col('price').filter(pl.col('price_level') == "bids[0].price").first()) / 2))
                                   ).sum() * pl.col('price').first()).alias('offer_liquidity_in_25bps'),
        ).sort('timestamp')
        
        orderbooks_liquidity = orderbooks_liquidity.with_columns(
            *[pl.col(col).ewm_mean(alpha=0.01, adjust=False).alias(col) for col in orderbooks_liquidity.columns if 'liquidity' in col]
        ).with_columns(
            *[pl.col(col) * -1 for col in orderbooks_liquidity.columns if 'bid' in col]
        )
        
        venue_stat = orderbooks_liquidity.mean().with_columns(
                pl.lit(f'{venue_name}').alias('exchange')
            )
        if venues_liquidity.height == 0:
            venues_liquidity = venue_stat
        else:
            venues_liquidity = pl.concat([venues_liquidity, venue_stat])
            
        print(f'{venue_name} is done')
    
    return venues_liquidity

In [ ]:
# SIGN - Apr 28; INIT - Apr 24; BABY - Apr 10; KAITO - Feb 25 ; IP - Feb 13
start_time_str = "2025-05-14"
end_time_str = "2025-05-14"
exchanges = ['binance-spot', 'okx-spot', 'bybit-spot', 'gate-spot',  'kucoin-spot'] # 'kraken-spot', 'kucoin',  'okx-spot', 'binance-perp',
data_types = ['book_snapshot_25']
price_ranges = [[0.0135, 0.0145], [0.0145, 0.0155], [0.0155, 0.0165], [0.0165, 0.0175]]

coin = 'PENGU'
venue_stats = []

for price_range in price_ranges:
    venue_stats.append(liquidity_examine(start_time_str, end_time_str, exchanges, [coin], price_range))

In [ ]:
venue_stats

In [ ]:
for (test, price_range) in zip(venue_stats, price_ranges):
    df = pd.DataFrame(test, columns=test.columns)

    plt.style.use('ggplot')

    # Set up the bar plot
    labels = df['exchange']
    x = range(len(labels))

    # Create subplots with 4 bars for each volume
    fig, ax = plt.subplots(figsize=(10, 6))

    # Bar width and positions
    bar_width = 0.1
    positions_buy_25bps = [i + 2 * bar_width for i in x]
    positions_sell_25bps = [i + 2 * bar_width for i in x]

    positions_buy_20bps = [i + 1 * bar_width for i in x]
    positions_sell_20bps = [i + 1 * bar_width for i in x]

    positions_buy_15bps = [i - 0 * bar_width for i in x]
    positions_sell_15bps = [i - 0 * bar_width for i in x]

    positions_buy_10bps = [i - 1 * bar_width for i in x]
    positions_sell_10bps = [i - 1 * bar_width for i in x]

    positions_buy_5bps = [i - 2 * bar_width for i in x]
    positions_sell_5bps = [i - 2 * bar_width for i in x]

    colors = sns.color_palette("muted", 10)

    # Plot each volume type
    # ax.bar(positions_buy_5bps, df['bid_liquidity_in_5bps'], bar_width, label='Bid Liquidity within 5bps', color = colors[0])
    # ax.bar(positions_sell_5bps, df['offer_liquidity_in_5bps'], bar_width, label='Offer Liquidity within 5bps', color = colors[1])
    # ax.bar(positions_buy_10bps, df['bid_liquidity_in_10bps'], bar_width, label='Bid Liquidity within 10bps', color = colors[2])
    # ax.bar(positions_sell_10bps, df['offer_liquidity_in_10bps'], bar_width, label='Offer Liquidity within 10bps', color = colors[3])
    # ax.bar(positions_buy_15bps, df['bid_liquidity_in_15bps'], bar_width, label='Bid Liquidity within 15bps', color = colors[4])
    # ax.bar(positions_sell_15bps, df['offer_liquidity_in_15bps'], bar_width, label='Offer Liquidity within 15bps', color = colors[5])
    # ax.bar(positions_buy_20bps, df['bid_liquidity_in_20bps'], bar_width, label='Bid Liquidity within 20bps', color = colors[6])
    # ax.bar(positions_sell_20bps, df['offer_liquidity_in_20bps'], bar_width, label='Offer Liquidity within 20bps', color = colors[7])
    # ax.bar(positions_buy_25bps, df['bid_liquidity_in_25bps'], bar_width, label='Bid Liquidity within 25bps')
    # ax.bar(positions_sell_25bps, df['offer_liquidity_in_25bps'], bar_width, label='Offer Liquidity within 25bps')
    
    ax.bar(positions_buy_5bps, df['bid_liquidity_in_5bps'] + df['offer_liquidity_in_5bps'], bar_width, label='Bid Liquidity within 5bps', color = colors[0])
    ax.bar(positions_buy_10bps, df['bid_liquidity_in_10bps'] + df['offer_liquidity_in_10bps'], bar_width, label='Bid Liquidity within 10bps', color = colors[1])
    ax.bar(positions_buy_15bps, df['bid_liquidity_in_15bps'] + df['offer_liquidity_in_15bps'], bar_width, label='Bid Liquidity within 15bps', color = colors[2])
    ax.bar(positions_buy_20bps, df['bid_liquidity_in_20bps'] + df['offer_liquidity_in_20bps'], bar_width, label='Bid Liquidity within 20bps', color = colors[3])
    

    # Add labels and title
    ax.set_xlabel('Exchanges')
    ax.set_ylabel('Volume')
    ax.set_title(f'Liquidity Comparison across Exchanges in Price Range ({price_range[0]}, {price_range[1]})')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    # Show plot
    plt.tight_layout()
    plt.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

for test, price_range in zip(venue_stats, price_ranges):
    df = pd.DataFrame(test, columns=test.columns)

    labels = df['exchange']

    fig = go.Figure()

    bar_width = 0.15

    offsets = [-1.5, -0.5, 0.5, 1.5]
    bps_labels = ['5bps', '10bps', '15bps', '20bps']
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

    for offset, bps_label, color in zip(offsets, bps_labels, colors):
        fig.add_trace(go.Bar(
            x=labels,
            y=df[f'bid_liquidity_in_{bps_label}'] + df[f'offer_liquidity_in_{bps_label}'],
            name=f'Ask-Bid Liquidity within {bps_label}',
            marker_color=color,
            offsetgroup=offset,
            width=bar_width
        ))

    fig.update_layout(
        title=f'Ask and Bid Liquidity Delta Accoss Different Spot Market on May 14th in Price Range ({price_range[0]}, {price_range[1]})',
        xaxis_title='Exchanges',
        yaxis_title='Volume',
        barmode='group',
        template='plotly_white',
        hovermode='closest',
        width=1500,  # Set the total width of the plot
        height=800,
    )

    fig.show()


In [ ]:
import plotly.graph_objects as go
import pandas as pd

for test, price_range in zip(venue_stats, price_ranges):
    df = pd.DataFrame(test, columns=test.columns)

    labels = df['exchange']

    fig = go.Figure()

    bar_width = 0.1
    offsets = [-0.2, -0.1, 0, 0.1]
    bps_labels = ['5bps', '10bps', '15bps', '20bps']
    colors_bid = ['#636EFA', '#00CC96', '#FFA15A', '#AB63FA']
    colors_offer = ['#636EFA', '#00CC96', '#FFA15A', '#AB63FA']

    for offset, bps_label, color_bid, color_offer in zip(offsets, bps_labels, colors_bid, colors_offer):
        fig.add_trace(go.Bar(
            x=[x + offset + 0.001 for x in range(len(labels))],
            y=df[f'bid_liquidity_in_{bps_label}'],
            name=f'Bid Liquidity {bps_label}',
            marker_color=color_bid,
            width=bar_width,
        ))
        fig.add_trace(go.Bar(
            x=[x + offset  for x in range(len(labels))],
            y=df[f'offer_liquidity_in_{bps_label}'],
            name=f'Offer Liquidity {bps_label}',
            marker_color=color_offer,
            width=bar_width,
        ))

    fig.update_layout(
        title=f'Accumulated Liquidity on both Bid and Ask side across Exchanges on May 14th in Price Range ({price_range[0]}, {price_range[1]})',
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(len(labels))),
            ticktext=labels
        ),
        yaxis_title='Volume',
        barmode='group',
        template='plotly_white',
        hovermode='closest',
        width=1500,  # Set the total width of the plot
        height=800,
    )

    fig.show()

### Test

In [ ]:
root_dir = './data'
api_key = 'TD.sSGNjEpohBl9i0FV.zz0-s-b6DZ9c2lt.uoNpgBPGaTITrA9.YlW26JKS0PSfoOG.2pEnnkHAnTpOloD.3wMo'

Tardis_data = TardisLoader(root_dir, api_key)

# Set time and exchanges
start_time = datetime.strptime('2025-05-13', '%Y-%m-%d')
end_time = datetime.strptime('2025-05-14', '%Y-%m-%d')
exchange = 'binance-futures' #'bybit-spot'， ‘binance-futures’
symbol = 'PENGUUSDT'
                    
# Load the data
trades_df = Tardis_data.read(start_time, end_time, data_type='trades', exchange=exchange, symbol=symbol)

# Take a look at the data
trades_df.describe()

In [ ]:
trades_df

In [ ]:
def resample_trades_data(raw_df: pl.DataFrame) -> pl.DataFrame():
    '''
    Step 1: Resample the raw transcation data into single trade (depends on the rule of exchanges)
    Step 2: Calculate basic features for each single trade, which will be used in aggregation features later
    Step 3: Filter significant trades by volume - to make sure that we always have aroud 100 valid data point per minute
    
    Single Trade Feature: 'vwap', 'volume', 'initial_price'
    Regime Feature (after filter): 'vwap_diff', 'vwap_diff_square'
    '''
    # Access to hyperparameters
    price_round = 6
    volume_round = 5
    latency_gap = 5_000
    
    # Feature of single trade (volume, vwap)
    grouped_trades = raw_df.with_columns(
        pl.when((pl.col('timestamp').diff() > latency_gap) ).then(pl.lit(1)).otherwise(0).cum_sum().fill_null(0).alias('bucket_id') # | (pl.col('side') != pl.col('side').shift())
    ).group_by("bucket_id").agg(
        pl.col("human_time").first().alias('start_human_time'),
        pl.col("timestamp").first().alias("start_exch_time"),
        pl.col("timestamp").last().alias("end_exch_time"),
        pl.col("price").filter(pl.col('side') == 'buy').first().alias("buy_start_price"),
        pl.col("price").filter(pl.col('side') == 'buy').last().alias("buy_end_price"),
        pl.col("price").filter(pl.col('side') == 'sell').first().alias("sell_start_price"),
        pl.col("price").filter(pl.col('side') == 'sell').last().alias("sell_end_price"),
        pl.col("amount").sum().alias("volume"),
        pl.col("amount").filter(pl.col('side') == 'buy').sum().alias("buy_volume"),
        pl.col("amount").filter(pl.col('side') == 'sell').sum().alias("sell_volume"),
        ((pl.col("price") * pl.col("amount")).sum() / pl.col("amount").sum()).alias("vwap")
    ).with_columns(
        pl.when(pl.col('buy_volume') >= pl.col('sell_volume')).then(pl.lit('buy')).otherwise(pl.lit('sell')).alias('side'),
        pl.when(pl.col('buy_volume') >= pl.col('sell_volume')).then(pl.col('buy_start_price')).otherwise(pl.col('sell_start_price')).alias('start_price'),
        pl.when(pl.col('buy_volume') >= pl.col('sell_volume')).then(pl.col('buy_end_price')).otherwise(pl.col('sell_end_price')).alias('end_price')
    )
    
    # Extra columns
    # grouped_trades = grouped_trades.with_columns(
    #     pl.when(pl.col('end_price') - pl.col('start_price') != 0).then(
    #         ((pl.col('vwap') - pl.col('start_price')) / (pl.col('end_price') - pl.col('start_price'))).round(3)
    #     ).otherwise(
    #         pl.lit(0)
    #     ).alias('vwap_pos')
    # )
        
    # Round digit for specific columns
    grouped_trades = grouped_trades.with_columns(
        pl.col('start_price').round(price_round),
        pl.col('end_price').round(price_round),
        pl.col('vwap').round(price_round),
        pl.col('volume').round(volume_round),
        pl.col('buy_volume').round(volume_round),
        pl.col('sell_volume').round(volume_round),
    ).sort('bucket_id')
        
    return grouped_trades

# grouped_df = resample_trades_data(binance_perp_btc, hyper_params_btc)

In [ ]:
grouped_trades = resample_trades_data(trades_df)

In [ ]:
all_user = grouped_trades.filter(pl.col('volume') >= grouped_trades['volume'].quantile(0.5))
large_user = grouped_trades.filter(pl.col('volume') >= grouped_trades['volume'].quantile(0.9))
medium_user = grouped_trades.filter((pl.col('volume') >= grouped_trades['volume'].quantile(0.5)) & (pl.col('volume') <= grouped_trades['volume'].quantile(0.9)))

# Calculate the decaying sum of buy volume

def cal_ls_ratio(grouped_df: pl.DataFrame, alpha_decay: float = 0.001) -> pl.DataFrame: 
    decaying_buy_volume_sum = 0
    decaying_buy_volume_sum_series = []

    # Calculate decaying sum iteratively
    for volume in grouped_df['buy_volume']:
        decaying_buy_volume_sum = volume + alpha_decay * decaying_buy_volume_sum
        decaying_buy_volume_sum_series.append(decaying_buy_volume_sum)

    # Calculate the decaying sum of buy volume
    decaying_sell_volume_sum = 0
    decaying_sell_volume_sum_series = []

    # Calculate decaying sum iteratively
    for volume in grouped_df['sell_volume']:
        decaying_sell_volume_sum = volume + alpha_decay * decaying_sell_volume_sum
        decaying_sell_volume_sum_series.append(decaying_sell_volume_sum)

    ema_feature_df = grouped_df.with_columns(
            pl.Series(decaying_buy_volume_sum_series).alias('buy_volume_decaying_sum'),
            pl.Series(decaying_sell_volume_sum_series).alias('sell_volume_decaying_sum'),
        ).with_columns(
        (pl.col('buy_volume_decaying_sum') / (pl.col('buy_volume_decaying_sum') + pl.col('sell_volume_decaying_sum'))).alias('ls_volume_ratio'),
    )
    
    return ema_feature_df[500:]

all_user_ls_fast = cal_ls_ratio(all_user, 0.9998)
large_user_ls_fast = cal_ls_ratio(large_user, 0.9998)
medium_user_ls_fast = cal_ls_ratio(medium_user, 0.9998)

all_user_ls_slow = cal_ls_ratio(all_user, 0.9999)
large_user_ls_slow = cal_ls_ratio(large_user, 0.9999)
medium_user_ls_slow = cal_ls_ratio(medium_user, 0.9999)

In [ ]:
# Assuming df1 and df2 are your two Polars DataFrames

df1 = large_user_ls_fast
df2 = medium_user_ls_fast
df3 = all_user_ls_fast

df4 = large_user_ls_slow
df5 = medium_user_ls_slow
df6 = all_user_ls_slow

# Create a figure
fig = go.Figure()

# Plot price from df1 on primary y-axis
fig.add_trace(go.Scatter(
    x=df1["start_human_time"].to_pandas(),
    y=df1["vwap"].to_pandas(),
    mode="lines",
    name="Price",
    yaxis="y1"
))

# # Plot ls_ratio from df1 on secondary y-axis
# fig.add_trace(go.Scatter(
#     x=df1["start_human_time"].to_pandas(),
#     y=df1["ls_volume_ratio"].to_pandas(),
#     mode="lines",
#     name="Large_User_LS_Ratio_Fast",
#     yaxis="y2"
# ))

# # Plot ls_ratio from df2 on secondary y-axis
# fig.add_trace(go.Scatter(
#     x=df2["start_human_time"].to_pandas(),
#     y=df2["ls_volume_ratio"].to_pandas(),
#     mode="lines",
#     name="Medium_User_LS_Ratio_Fast",
#     yaxis="y2"
# ))

# # Plot ls_ratio from df2 on secondary y-axis
# fig.add_trace(go.Scatter(
#     x=df3["start_human_time"].to_pandas(),
#     y=df3["ls_volume_ratio"].to_pandas(),
#     mode="lines",
#     name="Aggregated_LS_ratio_Fast",
#     yaxis="y2"
# ))

# # Plot ls_ratio from df1 on secondary y-axis
# fig.add_trace(go.Scatter(
#     x=df4["start_human_time"].to_pandas(),
#     y=df4["ls_volume_ratio"].to_pandas(),
#     mode="lines",
#     name="Large_User_LS_Ratio_Slow",
#     yaxis="y2"
# ))

# # Plot ls_ratio from df2 on secondary y-axis
# fig.add_trace(go.Scatter(
#     x=df5["start_human_time"].to_pandas(),
#     y=df5["ls_volume_ratio"].to_pandas(),
#     mode="lines",
#     name="Medium_User_LS_Ratio_Slow",
#     yaxis="y2"
# ))

# Plot ls_ratio from df2 on secondary y-axis
fig.add_trace(go.Scatter(
    x=df6["start_human_time"].to_pandas(),
    y=df6["ls_volume_ratio"].to_pandas(),
    mode="lines",
    name="Aggregated_LS_ratio",
    yaxis="y2"
))


# Update layout with two y-axes
fig.update_layout(
    title="Price and LS Ratios",
    xaxis=dict(
        title="Timestamp",
        rangeslider=dict(visible=True)
    ),
    yaxis=dict(
        title="Price",
        side="left"
    ),
    yaxis2=dict(
        title="LS Ratio",
        overlaying="y",
        side="right"
    ),
    template="plotly_white",
    width=2000,  # Set the total width of the plot
    height=700,
    hovermode="closest"  # Shows all data points at the same x position
)

# Show the plot
fig.show()